### ResNet18 Dataset Processing

##### Imports

In [1]:
from pathlib import Path
import shutil
import random
import cv2
import torch
from collections import Counter

##### Paths & Parameters

In [2]:
DATASET_DIR = Path("../data/raw/data")
OUTPUT_DIR = Path("../data/processed/resnet18_dataset")

CLASSES = ["need_cleaning", "free", "occupied", "awaiting"]

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MARGIN = 0.5  # 50% of width/height

OUTPUT_DIR.mkdir(exist_ok=True)

##### Crop Tables Using Ground Truth Boxes

In [3]:
def yolo_to_xyxy(label_line, img_w, img_h):
    cls, x, y, w, h = map(float, label_line.split())
    
    x1 = int((x - w/2) * img_w)
    y1 = int((y - h/2) * img_h)
    x2 = int((x + w/2) * img_w)
    y2 = int((y + h/2) * img_h)
    
    return x1, y1, x2, y2


for cls_name in CLASSES:
    img_dir = DATASET_DIR / cls_name / "images"
    lbl_dir = DATASET_DIR / cls_name / "labels"
    out_dir = OUTPUT_DIR / cls_name
    
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for img_path in img_dir.glob("*"):
        label_path = lbl_dir / img_path.with_suffix(".txt").name
        
        if not label_path.exists():
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        h, w = img.shape[:2]
        
        with open(label_path) as f:
            lines = f.readlines()
        
        for i, line in enumerate(lines):
            x1, y1, x2, y2 = yolo_to_xyxy(line, w, h)

            box_w = x2 - x1
            box_h = y2 - y1

            x_pad = int(box_w * MARGIN)
            y_pad = int(box_h * MARGIN)

            # Ensure coordinates stay within image boundaries
            x1_new = max(0, x1 - x_pad)
            y1_new = max(0, y1 - y_pad)
            x2_new = min(w, x2 + x_pad)
            y2_new = min(h, y2 + y_pad)

            crop = img[y1_new:y2_new, x1_new:x2_new]
            
            if crop.size == 0:
                continue
            
            crop_name = f"{img_path.stem}_crop{i}.jpg"
            cv2.imwrite(str(out_dir / crop_name), crop)

print("Cropping complete ✅")


Cropping complete ✅


##### Class Distribution Check

In [4]:
counts = {}

for cls_name in CLASSES:
    counts[cls_name] = len(list((OUTPUT_DIR / cls_name).glob("*")))

counts

{'need_cleaning': 405, 'free': 347, 'occupied': 126, 'awaiting': 82}

##### Balance Classes (Simple Undersampling)

In [6]:
min_count = min(counts.values())
print("Balancing to:", min_count)

balanced_dir = Path("../data/processed/resnet18_dataset_balanced")
balanced_dir.mkdir(exist_ok=True)

for cls_name in CLASSES:
    src_dir = OUTPUT_DIR / cls_name
    dst_dir = balanced_dir / cls_name
    
    dst_dir.mkdir(parents=True, exist_ok=True)
    
    images = list(src_dir.glob("*"))
    random.shuffle(images)
    
    # for img_path in images[:min_count]:
    for img_path in images:
        shutil.copy(img_path, dst_dir / img_path.name)

print("Balancing complete ✅")

Balancing to: 82
Balancing complete ✅
